# 02 — Topic Comparison

Compare LDA vs BERTopic topic models, examine coherence scores, and label topics.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import load_config
from src.data.load_ungdc import load_ungdc
from src.data.segment import segment_by_keywords
from src.data.preprocess import preprocess_corpus

cfg = load_config('../config.yaml')
corpus_df = load_ungdc(data_dir=str(cfg.data_dir), synthetic_mode=cfg.synthetic_mode,
                       year_start=cfg.focus_start, year_end=cfg.focus_end)
corpus_df = preprocess_corpus(corpus_df)
segments_df = segment_by_keywords(corpus_df, window_size=cfg.keyword_window_size)
if segments_df.empty:
    segments_df = corpus_df
print(f"Segments: {len(segments_df):,}")

In [ ]:
# LDA k=10
from src.analysis.topics_lda import train_lda, label_lda_topics, get_lda_topic_distributions

lda = train_lda(segments_df, k=10, random_seed=cfg.random_seed, passes=5)
labels = label_lda_topics(lda['model'])
for item in labels:
    print(f"Topic {item['topic_id']:2d} [{item['label']}]: {', '.join(item['top_words'][:8])}")

In [ ]:
# LDA k sweep (coherence)
from src.analysis.topics_lda import sweep_lda_k

k_range = [5, 10, 15, 20]
sweep = sweep_lda_k(segments_df, k_range=k_range, random_seed=cfg.random_seed, passes=3)
print(sweep)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sweep['k'], sweep['coherence_c_v'], 'o-', color='steelblue')
axes[0].set_xlabel('k (number of topics)')
axes[0].set_ylabel('Coherence (C_v)')
axes[0].set_title('LDA Coherence by k')
axes[1].plot(sweep['k'], sweep['perplexity'], 'o-', color='darkorange')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Log Perplexity')
axes[1].set_title('LDA Perplexity by k')
plt.tight_layout()
plt.show()

In [ ]:
# Topic distributions by country-year
topic_dist = get_lda_topic_distributions(
    lda['model'], lda['corpus'], lda['dictionary'], segments_df=segments_df
)
topic_dist.head()

In [ ]:
# Heatmap of topics over time
from src.viz.temporal import plot_topic_prevalence_heatmap
plot_topic_prevalence_heatmap(topic_dist)
plt.show()

In [ ]:
# BERTopic (optional)
from src.analysis.topics_bertopic import BERTOPIC_AVAILABLE, train_bertopic

if BERTOPIC_AVAILABLE:
    bt = train_bertopic(segments_df.head(3000), min_topic_size=10)
    if bt is not None:
        print(bt['topic_info'].head(10))
else:
    print('BERTopic not installed — skipping. Install with: pip install bertopic')

In [ ]:
# Intertopic distance map
from src.viz.topic_maps import plot_intertopic_distance_map
plot_intertopic_distance_map()
plt.show()